In [0]:
import logging
from pyspark.sql import functions as F
from pyspark.sql.window import Window

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("GoldOportunidadeMercado")

logger.info("Consolidando indicadores por UF...")

# 1. Métricas de Vendas / Emplacamentos (30 meses e últimos 6 meses)
df_emp_uf = (
    spark.table("silver_emplacamentos_veiculos")
    .groupBy("uf")
    .agg(
        F.sum("qtd_emplacamentos").alias("vendas_total_30m"),
        F.sum(F.when(F.col("tecnologia") == "BEV", F.col("qtd_emplacamentos")).otherwise(0)).alias("vendas_100_eletrico_30m"),
        F.sum(F.when(F.col("exige_recarga_externa"), F.col("qtd_emplacamentos")).otherwise(0)).alias("vendas_plugin_30m"),
        F.sum(F.when(F.col("ano_mes") >= "2026-05", F.col("qtd_emplacamentos")).otherwise(0)).alias("vendas_ultimos_6m")
    )
)

# 2. Métricas de Frota Circulante Atual (SENATRAN Out/2026)
df_frota_uf = (
    spark.table("silver_frota_senatran")
    .filter(F.col("uf") != "DESCONHECIDO")
    .groupBy("uf")
    .agg(
        F.sum("qtd_veiculos").alias("frota_total_eletrificados"),
        F.sum(F.when(F.col("exige_recarga_externa"), F.col("qtd_veiculos")).otherwise(0)).alias("frota_plugin_circulante")
    )
)

# 3. Infraestrutura de Eletropostos (Tabela Gold do 1º Pipeline)
df_postos_uf = (
    spark.table("gold_estacoes_estados")
    .select(
        F.col("estado").alias("uf"),
        F.col("qtde_estacoes").alias("qtde_eletropostos")
    )
)

# 4. Incentivo Tributário (IPVA 2026)
df_ipva_uf = (
    spark.table("silver_ipva_veiculos_2026")
    .select(
        "uf",
        "nome_estado",
        "regra_ipva_eletrico",
        "aliquota_eletrico_efetiva",
        "incentivo_fiscal_score"
    )
)

# Cruzamento Final (JOIN de todas as fontes)
df_gold_uf = (
    df_ipva_uf
    .join(df_emp_uf, on="uf", how="left")
    .join(df_frota_uf, on="uf", how="left")
    .join(df_postos_uf, on="uf", how="left")
    .fillna(0, subset=[
        "vendas_total_30m", "vendas_100_eletrico_30m", "vendas_plugin_30m",
        "vendas_ultimos_6m", "frota_total_eletrificados", "frota_plugin_circulante",
        "qtde_eletropostos"
    ])
    # Índice de Pressão de Recarga: quantos carros plug-in disputam cada eletroposto
    .withColumn(
        "carros_plugin_por_posto",
        F.when(F.col("qtde_eletropostos") > 0, F.round(F.col("frota_plugin_circulante") / F.col("qtde_eletropostos"), 1))
        .otherwise(F.col("frota_plugin_circulante").cast("double"))
    )
    # Participação nacional nas vendas recentes
    .withColumn(
        "share_vendas_nacional_pct",
        F.round((F.col("vendas_total_30m") / F.sum("vendas_total_30m").over(Window.partitionBy())) * 100, 2)
    )
    # Classificação Estratégica de Mercado para Expansão
    .withColumn(
        "diagnostico_oportunidade",
        F.when((F.col("vendas_ultimos_6m") >= 3000) & (F.col("regra_ipva_eletrico").isin("isento", "reduzido")),
               F.lit("1. Alta Demanda + Incentivo Fiscal (Mercado Maduro)"))
        .when((F.col("vendas_ultimos_6m") >= 1500) & (F.col("carros_plugin_por_posto") >= 250),
              F.lit("2. Forte Expansão com Gargalo de Recarga (Alta Oportunidade)"))
        .when(F.col("regra_ipva_eletrico").isin("isento", "reduzido"),
              F.lit("3. Incentivo Fiscal Favorável em Crescimento"))
        .otherwise(F.lit("4. Mercado em Consolidação / Sem Incentivo IPVA"))
    )
    .orderBy(F.desc("vendas_total_30m"))
)

df_gold_uf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_viabilidade_mercado_uf")
logger.info("Tabela 'gold_viabilidade_mercado_uf' criada com sucesso!")



In [0]:
logger.info("Refinando gold_ranking_modelos_ev com normalização de famílias de modelos...")

df_ficha_norm = (
    spark.table("silver_ficha_tecnica_veiculos")
    .withColumn("m_up", F.upper(F.trim(F.col("marca"))))
    .withColumn("mod_up", F.upper(F.trim(F.col("modelo"))))
    .withColumn(
        "marca_norm",
        F.when(F.col("m_up").isin("OMODA", "JAECOO"), F.lit("OMODA/JAECOO"))
        .otherwise(F.col("m_up"))
    )
    .withColumn(
        "modelo_norm",
        F.when(F.col("mod_up").contains("SONG"), F.lit("SONG (PRO + PLUS)"))
        .when(F.col("mod_up").contains("DOLPHIN") & (F.col("mod_up") != "DOLPHIN MINI"), F.lit("DOLPHIN"))
        .when((F.col("m_up") == "OMODA") & (F.col("mod_up") == "5 HEV"), F.lit("OMODA 5 HEV"))
        .otherwise(F.col("mod_up"))
    )
    .groupBy("marca_norm", "modelo_norm")
    .agg(
        F.round(F.avg("preco_referencia_brl"), 2).alias("preco_medio_brl"),
        F.round(F.avg("autonomia_inmetro_km"), 0).alias("autonomia_media_km"),
        F.first("carroceria").alias("carroceria")
    )
)

df_gold_modelos = (
    spark.table("silver_emplacamentos_veiculos")
    .groupBy("uf", "marca", "modelo", "tecnologia", "grupo_tecnologia")
    .agg(
        F.sum("qtd_emplacamentos").alias("total_emplacamentos"),
        F.sum(F.when(F.col("ano_mes") >= "2026-05", F.col("qtd_emplacamentos")).otherwise(0)).alias("emplacamentos_ultimos_6m")
    )
    .join(
        df_ficha_norm,
        (F.upper(F.trim(F.col("marca"))) == F.col("marca_norm")) & (F.upper(F.trim(F.col("modelo"))) == F.col("modelo_norm")),
        how="left"
    )
    .drop("marca_norm", "modelo_norm")
    .withColumn("vgc_estimado_brl", F.round(F.col("total_emplacamentos") * F.col("preco_medio_brl"), 2))
    .orderBy(F.desc("total_emplacamentos"))
)

df_gold_modelos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_ranking_modelos_ev")
